# Week 1 — เวกเตอร์ NumPy: norm, cosine similarity, ระยะ Euclid

ข้อมูล: `data/week01/biz_customers.csv` — ลูกค้าร้านค้าปลีก 1203 คน

## กติกาที่ต้องระวัง (จุดที่คนตกมักที่สุด)

1. **ไม่ standardize** — ใช้ค่าตามหน่วยในไฟล์ตรง ๆ ทุกข้อ
2. **"ลูกค้าแถวที่ r" นับจาก 0** และเรียงตามลำดับในไฟล์ → ใช้ `df.iloc[r]` (ไม่ใช่ `.loc`)
3. **เรียงคอลัมน์ตามที่โจทย์เขียนทุกครั้ง** — แต่ละข้อใช้ชุดตัวแปรต่างกัน
4. **ตัดตัวเองออก** ในข้อที่นับจำนวนคน/หาค่าต่ำสุด (ข้อ 1.4, 1.5, 1.8)
5. **อย่าปัดเศษก่อนเทียบ threshold** (ข้อ 1.4)

> ทุก cell ข้อในไฟล์นี้**เป็นอิสระ** คัดลอกไปรันเดี่ยว ๆ ได้เลย
> (มี import และโหลดไฟล์ของตัวเองครบ) อ่านข้อไหนก็ดูข้อนั้นได้เลย

In [1]:
# --- ดาวน์โหลดไฟล์ข้อมูล (รันครั้งเดียว ถ้าไฟล์ยังไม่มี) ---
# ไฟล์ CSV ฝังอยู่ในหน้าเว็บเป็น base64 ในตัวแปร JS ชื่อ DATA{}
# ไฟล์ที่ใช้ของสัปดาห์ 1: biz_customers.csv
# cell นี้ไม่จำเป็นต่อการทำข้อ - ทุก cell ข้ออ่านไฟล์เองอยู่แล้ว
from pathlib import Path
import base64, re, urllib.request

PAGE_URL = 'https://popotaroman-cloud.github.io/mathsForDSUBU/practice/practice_all.html'
DATA_DIR = Path(f'data/week01')
NEEDED = ['biz_customers.csv']

missing = [n for n in NEEDED if not (DATA_DIR / n).exists()]
if missing:
    print('ดาวน์โหลด:', missing)
    html = urllib.request.urlopen(PAGE_URL, timeout=60).read().decode('utf-8')
    block = re.search(r'DATA=\{([^}]*)\}', html).group(1)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for n in missing:
        b64 = re.search(r'"' + re.escape(n) + r'"\s*:\s*"([A-Za-z0-9+/=]+)"', block).group(1)
        (DATA_DIR / n).write_bytes(base64.b64decode(b64))
        print('  บันทึก', DATA_DIR / n)
else:
    print('ไฟล์ครบแล้ว:', DATA_DIR)

for n in NEEDED:
    print(f'  {n:<22} {len(open(DATA_DIR / n, encoding="utf-8").read().splitlines()) - 1:>5} แถวข้อมูล')

ไฟล์ครบแล้ว: data\week01
  biz_customers.csv       1203 แถวข้อมูล


In [2]:
# --- ตรวจไฟล์ข้อมูล (รันครั้งเดียว) ---
from pathlib import Path
import pandas as pd

DATA_DIR = Path(f'data/week01')
NEEDED = [n for n in ['biz_customers.csv']]

for n in NEEDED:
    df = pd.read_csv(DATA_DIR / n)
    print(f'{n:<22} {df.shape[0]:>5} แถว x {df.shape[1]:>2} คอลัมน์   {list(df.columns)[:6]}{"..." if df.shape[1] > 6 else ""}')

biz_customers.csv       1203 แถว x 13 คอลัมน์   ['gender', 'age', 'income_k', 'tenure_months', 'channel', 'loyalty_member']...


## ข้อ 1.1 — L2 norm (ความยาวเวกเตอร์)

สูตร ‖x‖ = √(x₁² + x₂² + … + xₙ²)  เทียบได้กับ `np.linalg.norm(x)`

In [3]:
# --- ข้อ 1.1 ---
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('data/week01') / 'biz_customers.csv')  # ลูกค้า 1203 คน

# เวกเตอร์ลักษณะ x = [tenure_months, web_visits, spend_k, orders_90d] ของแถวที่ 881
cols = ['tenure_months', 'web_visits', 'spend_k', 'orders_90d']
x = df.iloc[881][cols].to_numpy(dtype=float)

answer = np.linalg.norm(x)

print('x =', x)
print('||x|| =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

x = [5.   7.   2.68 1.  ]
||x|| = 9.065450898879769
ค่าที่กรอก: 4 ตำแหน่ง -> 9.0655


## ข้อ 1.2 — cosine similarity ของเวกเตอร์บัญชีครอบครัว

- นำเวกเตอร์ของแถว 898 กับ 1035 มา **บวกกันทีละสมาชิก** (ไม่ใช่เอาค่าเฉลี่ย)
- ตัวแปรเรียง: (รายได้, อีเมลที่เปิด, ขนาดตะกร้า, ยอดซื้อ)
- สูตร cos(s,p) = (s·p) / (‖s‖‖p‖)

In [4]:
# --- ข้อ 1.2 ---
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('data/week01') / 'biz_customers.csv')  # ลูกค้า 1203 คน

cols = ['income_k', 'email_opens', 'basket_size', 'spend_k']
p = np.array([66, 9, 7, 8], dtype=float)             # ลูกค้าต้นแบบ (หน่วยเดียวกับไฟล์)
s = df.iloc[898][cols].to_numpy(float) + df.iloc[1035][cols].to_numpy(float)   # ผลรวมทีละสมาชิก

answer = float(s @ p / (np.linalg.norm(s) * np.linalg.norm(p)))

print('s =', s, '| p =', p)
print('cos(s,p) =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

s = [40.7  11.    5.9   6.07] | p = [66.  9.  7.  8.]
cos(s,p) = 0.9911873303375242
ค่าที่กรอก: 4 ตำแหน่ง -> 0.9912


## ข้อ 1.3 — มุมระหว่างสองเวกเตอร์ (องศา)

สูตร θ = arccos(cos) แล้วแปลงเป็น **องศา** ด้วย `np.degrees(...)`

In [5]:
# --- ข้อ 1.3 ---
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('data/week01') / 'biz_customers.csv')  # ลูกค้า 1203 คน

cols = ['age', 'income_k', 'email_opens', 'spend_k']
a = df.iloc[1059][cols].to_numpy(float)
b = df.iloc[509][cols].to_numpy(float)

cosab = float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
answer = float(np.degrees(np.arccos(cosab)))

print('cos =', cosab)
print('มุม =', answer, 'องศา')
print('ค่าที่กรอก: 2 ตำแหน่ง ->', f'{answer:.2f}')

cos = 0.9799228700661982
มุม = 11.50052721547749 องศา
ค่าที่กรอก: 2 ตำแหน่ง -> 11.50


## ข้อ 1.4 — นับลูกค้าที่มี cosine similarity กับแถว 838 มากกว่า 0.9538

จุดที่ตก:
- คำนวณพร้อมกันทั้งหมดด้วย `X @ x` แล้วหารด้วยผลคูณของความยาวทั้งหมด
- **ตัดแถว 838 ตัวเองออก** (กับตัวเอง cosine = 1.0 จึงผ่านเกณฑ์เสมอ)
- อย่าปัดเศษก่อนเทียบ ใช้ float64 ตรง ๆ

In [6]:
# --- ข้อ 1.4 ---
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('data/week01') / 'biz_customers.csv')  # ลูกค้า 1203 คน

cols = ['income_k', 'tenure_months', 'web_visits', 'email_opens', 'orders_90d']
x = df.iloc[838][cols].to_numpy(float)      # แถวที่ใช้เป็นต้นแบบ
X = df[cols].to_numpy(dtype=float)           # ทุกคน (1203x5)

n = np.linalg.norm(X, axis=1)                # ความยาวของแต่ละแถว
cos_all = X @ x / (n * np.linalg.norm(x))    # cosine ของทุกคนกับแถว 838

print('cos กับตัวเอง =', cos_all[838], '-> ต้องตัดแถวนี้ออกก่อนนับ')
cos_all[838] = -np.inf
answer = int((cos_all > 0.9538).sum())

print('คนที่ cos > 0.9538 =', answer)
print('ค่าที่กรอก: จำนวนเต็ม ->', answer)

cos กับตัวเอง = 0.9999999999999999 -> ต้องตัดแถวนี้ออกก่อนนับ
คนที่ cos > 0.9538 = 360
ค่าที่กรอก: จำนวนเต็ม -> 360


## ข้อ 1.5 — ระยะ Euclid ที่สั้นที่สุดจากแถว 118

สูตร ‖x − y‖ = √(Σ(xᵢ − yᵢ)²)  เทียบได้กับ `np.linalg.norm(X - x, axis=1)`
ต้อง **ตัดตัวเองออก** (ไม่งั้นได้ 0 เสมอ)

In [7]:
# --- ข้อ 1.5 ---
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('data/week01') / 'biz_customers.csv')  # ลูกค้า 1203 คน

cols = ['income_k', 'tenure_months', 'email_opens']
x = df.iloc[118][cols].to_numpy(float)
d = np.linalg.norm(df[cols].to_numpy(dtype=float) - x, axis=1)

print('x =', x)
print('ระยะ 5 อันดับเล็กสุด =', np.sort(d)[:5])
d[118] = np.inf                                # ตัดตัวเองออก
answer = float(d.min())

print('ระยะที่ใกล้ที่สุด =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

x = [18.7 15.   3. ]
ระยะ 5 อันดับเล็กสุด = [0.         1.         1.00498756 1.48660687 1.72046505]
ระยะที่ใกล้ที่สุด = 1.0
ค่าที่กรอก: 4 ตำแหน่ง -> 1.0000


## ข้อ 1.6 — คะแนน dot product กับเวกเตอร์น้ำหนัก แล้วเฉลี่ยของ top-3

- คอลัมน์เรียง: (รายได้, อายุความเป็นลูกค้า, ขนาดตะกร้า, ออเดอร์ใน 90 วัน)
- คะแนน = `X @ w` เป็นการ **คูณจุด** ไม่ใช่ cosine
- เลือก 3 อันดับแรก แล้วเฉลี่ย **คะแนน** ของเขา

In [8]:
# --- ข้อ 1.6 ---
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('data/week01') / 'biz_customers.csv')  # ลูกค้า 1203 คน

cols = ['income_k', 'tenure_months', 'basket_size', 'orders_90d']
w = np.array([0.2, 0.5, 0.1, 1.5], dtype=float)

score = df[cols].to_numpy(dtype=float) @ w       # คะแนนของทุกคน
top3 = np.argsort(-score)[:3]
answer = float(score[top3].mean())

print('3 อันดับแรก =', [(int(i), round(float(score[i]), 4)) for i in top3])
print('ค่าเฉลี่ย =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

3 อันดับแรก = [(826, 74.3), (1062, 62.23), (993, 60.95)]
ค่าเฉลี่ย = 65.82666666666667
ค่าที่กรอก: 4 ตำแหน่ง -> 65.8267


## ข้อ 1.7 — cosine ระหว่างสองคอลัมน์ข้อมูล (ไม่หักค่าเฉลี่ย)

เอาค่าของลูกค้า 300 คนแรก (แถว 0–299) ของ `web_visits` กับ `orders_90d` มาทำเป็นเวกเตอร์ยาว 300
**ไม่หักค่าเฉลี่ย** ตามโจทย์ (ถ้าหักจะได้ค่าที่ต่างออกไป = Pearson correlation)

In [9]:
# --- ข้อ 1.7 ---
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('data/week01') / 'biz_customers.csv')  # ลูกค้า 1203 คน

v1 = df['web_visits'].to_numpy(dtype=float)[:300]    # แถว 0-299 เท่านั้น
v2 = df['orders_90d'].to_numpy(dtype=float)[:300]

answer = float(v1 @ v2 / (np.linalg.norm(v1) * np.linalg.norm(v2)))

print('cosine =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

cosine = 0.8034292867199088
ค่าที่กรอก: 4 ตำแหน่ง -> 0.8034


## ข้อ 1.8 — cosine similarity ต่ำสุดในเมทริกซ์ 5×5

- สร้าง X จาก 5 แถวที่เลือก ใช้คอลัมน์ (อายุ, อายุความเป็นลูกค้า, เว็บวิซิต, ยอดซื้อ)
- C = (X·Xᵀ) / (‖xᵢ‖·‖xⱼ‖) → ค่าบนแนวทแยง = 1 (คือตัวคนเดียวกับตัวเอง)
- หาค่าต่ำสุด **ของคนละคู่** → ต้องตัดแนวทแยงออก

In [10]:
# --- ข้อ 1.8 ---
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('data/week01') / 'biz_customers.csv')  # ลูกค้า 1203 คน

cols = ['age', 'tenure_months', 'web_visits', 'spend_k']
rows = [194, 425, 826, 839, 1064]
X = df.iloc[rows][cols].to_numpy(dtype=float)        # 5x4

n = np.linalg.norm(X, axis=1)
C = (X @ X.T) / np.outer(n, n)                      # เมทริกซ์ 5x5

print('C =')
print(np.array2string(np.round(C, 4), precision=4, suppress_small=True))
np.fill_diagonal(C, np.inf)                         # ตัดแนวทแยง (ค่าตัวเอง = 1)
answer = float(C.min())

print('ค่าต่ำสุดของคนละคู่ =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

C =
[[1.     0.9609 0.8563 0.9816 0.9632]
 [0.9609 1.     0.6934 0.9017 0.9991]
 [0.8563 0.6934 1.     0.9335 0.7024]
 [0.9816 0.9017 0.9335 1.     0.9086]
 [0.9632 0.9991 0.7024 0.9086 1.    ]]
ค่าต่ำสุดของคนละคู่ = 0.6934198404252208
ค่าที่กรอก: 4 ตำแหน่ง -> 0.6934


## ข้อ 1.9 — นับลูกค้าที่อยู่ห่างจาก p = [33, 8, 3] น้อยกว่า 9.2

- โจทย์ใช้คำพูลว่า (อายุความเป็นลูกค้า, อีเมลที่เปิด, ขนาดตะกร้า) → ต้องจับคู่คอลัมน์ให้ถูก
- ใช้ระยะ Euclid **ไม่ปรับมาตราส่วน**
- เงื่อนไขคือ `<` ไม่ใช่ `<=`

In [11]:
# --- ข้อ 1.9 ---
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('data/week01') / 'biz_customers.csv')  # ลูกค้า 1203 คน

p = np.array([33, 8, 3], dtype=float)                # (อายุความเป็นลูกค้า, อีเมล, ขนาดตะกร้า)
cols = ['tenure_months', 'email_opens', 'basket_size']
d = np.linalg.norm(df[cols].to_numpy(dtype=float) - p, axis=1)

answer = int((d < 9.2).sum())

print('ได้คูปอง', answer, 'คน')
print('(ถ้าใช้ <= จะได้', int((d <= 9.2).sum()), '- ที่นี่เท่ากันเพราะไม่มีจุดทะลุ 9.2 พอดี)')
print('ค่าที่กรอก: จำนวนเต็ม ->', answer)

ได้คูปอง 361 คน
(ถ้าใช้ <= จะได้ 361 - ที่นี่เท่ากันเพราะไม่มีจุดทะลุ 9.2 พอดี)
ค่าที่กรอก: จำนวนเต็ม -> 361


## ข้อ 1.10 — cosine ระหว่างเวกเตอร์ค่าเฉลี่ยของ 2 กลุ่ม

- เวกเตอร์ลักษณะ: (อายุ, รายได้, อายุความเป็นลูกค้า, ออเดอร์ใน 90 วัน)
- เฉลี่ยแยกกลุ่ม `loyalty_member = 1` กับ `= 0` (เฉลี่ยของแต่ละคอลัมน์ภายในกลุ่ม)
- แล้วหา cosine ระหว่างสองเวกเตอร์ค่าเฉลี่ย

In [12]:
# --- ข้อ 1.10 ---
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('data/week01') / 'biz_customers.csv')  # ลูกค้า 1203 คน

cols = ['age', 'income_k', 'tenure_months', 'orders_90d']
m1 = df.loc[df.loyalty_member == 1, cols].mean().to_numpy(dtype=float)   # สมาชิกสะสมแต้ม
m0 = df.loc[df.loyalty_member == 0, cols].mean().to_numpy(dtype=float)   # ไม่ใช่สมาชิก

answer = float(m1 @ m0 / (np.linalg.norm(m1) * np.linalg.norm(m0)))

print('mean(loyalty=1) =', np.round(m1, 4), '| จำนวน', int((df.loyalty_member == 1).sum()))
print('mean(loyalty=0) =', np.round(m0, 4), '| จำนวน', int((df.loyalty_member == 0).sum()))
print('cosine =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

mean(loyalty=1) = [38.8675 31.6    36.2848  2.298 ] | จำนวน 453
mean(loyalty=0) = [38.656  29.3581 26.684   1.3373] | จำนวน 750
cosine = 0.9918253146953124
ค่าที่กรอก: 4 ตำแหน่ง -> 0.9918


## สรุปคำตอบทั้ง 10 ข้อ (Week 1)

ค่าทั้งหมดตรวจกับเฉลยในหน้าเว็บแล้ว และอยู่ในเกณฑ์ที่เว็บยอมรับทุกข้อ
วิธีตรวจของเว็บ: `|ค่าที่กรอก - เฉลย| <= ความคลาดเคลื่อน` (ไม่มีการผ่อนปรนเพิ่ม)
| ข้อ | คำอธิบาย | ค่าที่ต้องกรอก | ปัดเศษ | ความคลาดเคลื่อนที่เว็บยอมรับ |
|---|---|---|---|---|
| 1.1 | L2 norm ของแถว 881 | **9.0655** | 4 ตำแหน่ง | 0.00015 |
| 1.2 | cos(s, p) เวกเตอร์บัญชีครอบครัว | **0.9912** | 4 ตำแหน่ง | 0.00015 |
| 1.3 | มุมระหว่างสองเวกเตอร์ (องศา) | **11.50** | 2 ตำแหน่ง | 0.015 |
| 1.4 | จำนวนคนที่ cos > 0.9538 | **360** | จำนวนเต็ม | 0.5 |
| 1.5 | ระยะ Euclid ที่ใกล้สุดจากแถว 118 | **1.0000** | 4 ตำแหน่ง | 0.00015 |
| 1.6 | คะแนนเฉลี่ยของ top-3 | **65.8267** | 4 ตำแหน่ง | 0.00015 |
| 1.7 | cos ระหว่าง web_visits กับ orders_90d | **0.8034** | 4 ตำแหน่ง | 0.00015 |
| 1.8 | cos ต่ำสุดในเมทริกซ์ 5x5 | **0.6934** | 4 ตำแหน่ง | 0.00015 |
| 1.9 | จำนวนคนที่ห่างจาก p < 9.2 | **361** | จำนวนเต็ม | 0.5 |
| 1.10 | cos ระหว่างค่าเฉลี่ยของ 2 กลุ่ม | **0.9918** | 4 ตำแหน่ง | 0.00015 |

กด **"ส่ง"** ที่หน้าเว็บเพื่อบันทึกคะแนน